# Visualizing Optimizer Path on 2D Loss Landscape

In this tutorial, we will use **Nanograd** to visualize how different optimizers (**SGD** vs **Adam**) navigate a tricky 2D loss surface. We will:
1. Define the mathematical function (Beale's Function) which has a sharp, narrow valley.
2. Perform gradient descent using `SGD` and `Adam` starting from the same initial coordinates.
3. Track and compare their optimization trajectories.
4. Plot the paths on top of a 2D contour map.

---

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt

from nanograd import Tensor
from nanograd.optim import SGD, Adam

## 1. Beale's Function

Beale's function is a standard benchmark function used to test optimization algorithms. It is defined as:
$$f(x, y) = (1.5 - x + xy)^2 + (2.25 - x + xy^2)^2 + (2.625 - x + xy^3)^2$$
It has a global minimum at $(3.0, 0.5)$ which lies in a flat, narrow valley.

In [ ]:
def beale_fn(x, y):
    term1 = (1.5 - x + x * y) ** 2
    term2 = (2.25 - x + x * (y ** 2)) ** 2
    term3 = (2.625 - x + x * (y ** 3)) ** 2
    return term1 + term2 + term3

## 2. Tracking Optimization Paths

In [ ]:
def optimize_path(optimizer_cls, start_pos, epochs=300, lr=0.01):
    # Initialize variables as Tensors
    x = Tensor(np.array([start_pos[0]]))
    y = Tensor(np.array([start_pos[1]]))
    
    # Instantiate optimizer
    if optimizer_cls == SGD:
        opt = SGD([x, y], learning_rate=lr)
    else:
        opt = Adam([x, y], learning_rate=lr)
        
    path = [(x.data.item(), y.data.item())]
    
    for epoch in range(epochs):
        # Compute loss
        loss = beale_fn(x, y)
        
        # Backward pass
        opt.zero_grad()
        loss.backward()
        
        # Update coordinates
        opt.step()
        
        path.append((x.data.item(), y.data.item()))
        
    return np.array(path)

start_point = (1.0, 1.5)
epochs = 400

# Run SGD with learning rate 0.0005 (SGD easily explodes on Beale's function if too high!)
sgd_path = optimize_path(SGD, start_point, epochs=epochs, lr=0.0005)

# Run Adam with learning rate 0.05 (Adam adapts learning rates, so it can handle 100x larger lr)
adam_path = optimize_path(Adam, start_point, epochs=epochs, lr=0.05)

## 3. Plotting the Trajectories on 2D Contour Map

In [ ]:
# Generate grid for contour map
x_grid = np.linspace(-0.5, 4.5, 300)
y_grid = np.linspace(-1.5, 2.0, 300)
X, Y = np.meshgrid(x_grid, y_grid)
Z = beale_fn(X, Y)

plt.figure(figsize=(10, 8))
# Log-scale contours for better visibility of Beale's function values
plt.contour(X, Y, Z, levels=np.logspace(0, 5, 25), cmap='plasma', alpha=0.6)
plt.plot(sgd_path[:, 0], sgd_path[:, 1], color='#e377c2', label='SGD Path (lr=0.0005)', linewidth=2.5, marker='.')
plt.plot(adam_path[:, 0], adam_path[:, 1], color='#1f77b4', label='Adam Path (lr=0.05)', linewidth=2.5, marker='.')
plt.plot(3.0, 0.5, 'r*', markersize=15, label='Global Min (3.0, 0.5)', edgecolors='k')
plt.plot(start_point[0], start_point[1], 'go', markersize=10, label='Start Point', edgecolors='k')

plt.title("SGD vs Adam on Beale's Loss Surface")
plt.xlabel('x')
plt.ylabel('y')
plt.xlim(-0.5, 4.5)
plt.ylim(-1.5, 2.0)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()

## Observations

- **SGD (pink)** is very slow to progress in the flat regions and can easily oscillate out of control if the learning rate is even slightly increased. This is because its step size relies purely on the local gradient magnitude, which vanishes in flat valleys.
- **Adam (blue)** quickly navigates the plateau, adapts its step size dynamically using running moment estimates, and directly converges to the global minimum (star).